# v0.3 只读复核
从本 Notebook 所在目录打开。所有SQL以mode=ro连接研究库；不包含训练或写库代码。已保存本次实际执行输出。

旧问题表保留原状，当前结论请读issues_effective；有效窗口请读endpoint_effective_review。

In [1]:
from pathlib import Path
import sqlite3, json
root=Path.cwd()
c=sqlite3.connect((root/'cycling_v0_3.sqlite').as_uri()+'?mode=ro',uri=True)
c.row_factory=sqlite3.Row
print(c.execute('PRAGMA integrity_check').fetchone()[0])

ok


In [2]:
print(json.dumps([dict(r) for r in c.execute('SELECT status,count(*) AS endpoints FROM admissions GROUP BY status')],ensure_ascii=False,indent=2))
print('training_ready',c.execute('SELECT count(*) FROM training_ready').fetchone()[0])

[
  {
    "status": "candidate_pending_final_record_audit",
    "endpoints": 54
  },
  {
    "status": "conditional_hold",
    "endpoints": 16
  },
  {
    "status": "frozen_pending_clarification",
    "endpoints": 18
  },
  {
    "status": "outside_primary_scope",
    "endpoints": 1
  }
]
training_ready 0


In [3]:
print(json.dumps([dict(r) for r in c.execute("SELECT record_id,voltage_lower_V,voltage_upper_V FROM endpoint_effective_review WHERE paper_id='R107'")],ensure_ascii=False,indent=2))

[
  {
    "record_id": "R107_CY01",
    "voltage_lower_V": 0.0,
    "voltage_upper_V": 1.0
  },
  {
    "record_id": "R107_CY02",
    "voltage_lower_V": null,
    "voltage_upper_V": null
  }
]


In [4]:
print(json.dumps([dict(r) for r in c.execute("SELECT issue_id,record_id,status,effective_status,resolution_review_id FROM issues_effective WHERE code='batch1_voltage_window_consistency'")],ensure_ascii=False,indent=2))

[
  {
    "issue_id": 235,
    "record_id": "R112_CY02",
    "status": "open",
    "effective_status": "resolved_by_visual_erratum",
    "resolution_review_id": 97
  },
  {
    "issue_id": 237,
    "record_id": "R112_CY03",
    "status": "open",
    "effective_status": "resolved_by_visual_erratum",
    "resolution_review_id": 102
  }
]


In [5]:
print(json.dumps([dict(r) for r in c.execute("SELECT record_id,current_density_A_g,pore_unit_review_required,total_pore_volume_cm3_g FROM endpoint_review WHERE paper_id='R091'")],ensure_ascii=False,indent=2))
print(json.dumps([dict(r) for r in c.execute('SELECT * FROM validation_links')],ensure_ascii=False,indent=2))
c.close()

[
  {
    "record_id": "R091_CY01",
    "current_density_A_g": 3.0,
    "pore_unit_review_required": 1,
    "total_pore_volume_cm3_g": null
  }
]
[
  {
    "link_id": 1,
    "record_a": "R107_CY01",
    "record_b": "R107_CY02",
    "related_doi": null,
    "relation": "shared_provisional_cycle_test",
    "status": "pending",
    "group_policy": "must_share_group",
    "rationale": "No new test or independent material invented.",
    "review_id": 109
  },
  {
    "link_id": 2,
    "record_a": "SRC123_CY02",
    "record_b": null,
    "related_doi": "10.1016/j.colsurfa.2016.10.009",
    "relation": "prior_material_or_curve_overlap",
    "status": "pending",
    "group_policy": "if_ingested_union_until_resolved",
    "rationale": "Cannot assign an independent test group to a potentially reused predecessor.",
    "review_id": 112
  },
  {
    "link_id": 3,
    "record_a": "R001_CY01",
    "record_b": null,
    "related_doi": "10.1002/asia.202400622",
    "relation": "prior_material_or_curve